### Example Fixed Project

The rest of this tutorial uses pre compiled ORBIT configs that are stored as .yaml files in the '~/configs/ folder. There are load and save methods available in ORBIT for working with .yaml files. These example projects each exhibit different functionalities within ORBIT. Using these examples and combinations of them, most project configurations can be modeled. 

In [1]:
import os
import pandas as pd
from ORBIT import ProjectManager, load_config

weather = pd.read_csv("data/era5_41.0N_125.0W_1989_2019.csv", parse_dates=["datetime"])\
            .set_index("datetime")

### Load the project configuration

In [2]:
fixed_config = load_config("configs/example_floating_project_600MW.yaml")  # Configs can be loaded with absolute or relative paths

print(type(fixed_config))                                         # They are loaded in as dictionaries.

print(f"Num turbines: {fixed_config['plant']['num_turbines']}")   # Once a configuration is loaded, different parameters can  
print(f"Turbine: {fixed_config['turbine']}")                      # be accessed using dict access.
print(f"\nSite: {fixed_config['site']}")

<class 'dict'>
Num turbines: 50
Turbine: 12MW_generic

Site: {'depth': 739, 'distance': 189, 'distance_to_landfall': 36, 'mean_windspeed': 8.41}


### Phases

This fixed project represents a generic Offshore Wind farm with 50 6MW turbines. It includes 5 design modules and 6 installation modules as seen below. This is a common set of modules to run for a fixed bottom project. This config will model the procurement and installation of monopiles, scour protection, array system, export system, offshore substation and the turbines.

In [3]:
print(f"Design phases: {fixed_config['design_phases']}")
print(f"\nInstall phases: {list(fixed_config['install_phases'].keys())}")

Design phases: ['CustomArraySystemDesign', 'ElectricalDesign', 'MooringSystemDesign', 'OffshoreFloatingSubstationDesign', 'SemiSubmersibleDesign']

Install phases: ['ArrayCableInstallation', 'ExportCableInstallation', 'MooredSubInstallation', 'MooringSystemInstallation', 'FloatingSubstationInstallation']


### Run

This project is always being modeled with the example weather project supplied that is representative of US East Coast wind farm locations.

In [4]:
project = ProjectManager(fixed_config, weather=weather)
project.run()

ORBIT library intialized at 'C:\ORBIT_procurement_by_year\ORBIT-2021\ORBIT\library'


Missing data in columns ['bury_speed']; all values will be calculated.DeprecationWarning: C:\ORBIT_procurement_by_year\ORBIT-2021\ORBIT\ORBIT\manager.py:730
landfall dictionary will be deprecated and moved into [export_system_design][landfall].DeprecationWarning: C:\ORBIT_procurement_by_year\ORBIT-2021\ORBIT\ORBIT\phases\design\_cables.py:417
landfall dictionary will be deprecated and moved into [export_system][landfall].DeprecationWarning: C:\ORBIT_procurement_by_year\ORBIT-2021\ORBIT\ORBIT\phases\install\quayside_assembly_tow\moored.py:94
support_vessel will be deprecated and replaced with towing_vessels and ahts_vessel in the towing groups.
['towing_vessl_groups]['station_keeping_vessels'] will be deprecated and replaced with ['towing_vessl_groups]['ahts_vessels'].
Offshore substation substructure is Monopile and should be 'Floating'.


### Top Level Outputs

ProjectManager offers several high level result categories:
- Installation CapEx
- System CapEx (procurement of BOS subcomponents)
- Turbine CapEx
- Soft CapEx (project management costs)
- Total CapEx
- Total installation time
- etc.

In [5]:
print(f"Installation CapEx:  {project.installation_capex/1e6:.0f} M")
print(f"System CapEx:        {project.system_capex/1e6:.0f} M")
print(f"Turbine CapEx:       {project.turbine_capex/1e6:.0f} M")
print(f"Soft CapEx:          {project.soft_capex/1e6:.0f} M")
print(f"Total CapEx:        {project.total_capex/1e6:.0f} M")

print(f"\nInstallation Time: {project.installation_time:.0f} h")

Installation CapEx:  327 M
System CapEx:        1664 M
Turbine CapEx:       982 M
Soft CapEx:          638 M
Total CapEx:        3771 M

Installation Time: 24502 h


### CapEx Breakdown

In [6]:
# The breakdown of project costs by module is available  at 'capex_breakdown'
data = project.capex_detailed_soft_capex_breakdown_per_kw

# Convert the dictionary into a pandas DataFrame
df = pd.DataFrame(list(data.items()), columns=['Category', 'Value'])

# Add a "Total" row
total_row = pd.DataFrame([['Total', df['Value'].sum()]], columns=['Category', 'Value'])
df = pd.concat([df, total_row], ignore_index=True)

# Display the DataFrame
df.to_csv("floating_costs.csv")
print(df)

                            Category        Value
0                       Array System   291.405834
1                      Export System   363.363031
2                       Substructure  1233.362619
3                     Mooring System   528.093628
4                Offshore Substation   357.524099
5          Array System Installation   204.171732
6         Export System Installation    88.148516
7          Substructure Installation   128.902156
8        Mooring System Installation   109.831460
9   Offshore Substation Installation    13.561698
10                           Turbine  1637.000000
11                           Project   267.181600
12            Construction Insurance    60.059283
13                   Decommissioning    95.307723
14                     Commissioning    60.059283
15           Procurement Contingency   268.981022
16          Installation Contingency   187.892369
17            Construction Financing   390.898523
18                             Total  6285.744577


### Installation Actions

In [7]:
df = pd.DataFrame(project.actions)    # The project simulation logs are also available for all modules
df

,cost_multiplier,agent,action,duration,cost,level,time,phase,location,phase_name,max_waveheight,max_windspeed,transit_speed,num_vessels,num_ahts_vessels
0,0.5,Array Cable Installation Vessel,Mobilize,72.000000,3.124920e+05,ACTION,0.000000,ArrayCableInstallation,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0.5,Export Cable Installation Vessel,Mobilize,72.000000,3.124920e+05,ACTION,0.000000,ExportCableInstallation,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,Onshore Construction,Onshore Construction,0.000000,4.001844e+06,ACTION,0.000000,ExportCableInstallation,Landfall,NaN,NaN,NaN,NaN,NaN,NaN
3,1.0,Mooring System Installation Vessel,Mobilize,168.000000,7.419230e+05,ACTION,0.000000,MooringSystemInstallation,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,Substation Assembly Line 1,Substation Substructure Assembly,0.000000,0.000000e+00,ACTION,0.000000,FloatingSubstationInstallation,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2621,NaN,Array Cable Installation Vessel,Lay/Bury Cable,133.210311,1.156310e+06,ACTION,8344.670115,ArrayCableInstallation,NaN,ArrayCableInstallation,2.0,25.0,11.5,NaN,NaN
2622,NaN,Array Cable Installation Vessel,Prepare Cable,1.000000,8.680333e+03,ACTION,8345.670115,ArrayCableInstallation,NaN,ArrayCableInstallation,NaN,NaN,NaN,NaN,NaN
2623,NaN,Array Cable Installation Vessel,Pull In Cable,5.500000,4.774183e+04,ACTION,8351.170115,ArrayCableInstallation,NaN,ArrayCableInstallation,NaN,NaN,NaN,NaN,NaN
2624,NaN,Array Cable Installation Vessel,Terminate Cable,5.500000,4.774183e+04,ACTION,8356.670115,ArrayCableInstallation,NaN,ArrayCableInstallation,NaN,NaN,NaN,NaN,NaN


In [8]:
# These logs can be sorted by phase by using DataFrame operations

turbine_install = df.loc[df['phase']=="TurbineInstallation"]
turbine_install

,cost_multiplier,agent,action,duration,cost,level,time,phase,location,phase_name,max_waveheight,max_windspeed,transit_speed,num_vessels,num_ahts_vessels


In [9]:
# Operations can also be grouped to see a total amount of time spend on each operation

turbine_install.groupby(["action"]).sum()['duration']

Series([], Name: duration, dtype: float64)